# 03 - Feature Engineering
## MA Waterways Heatwave Risk Analysis

**Objective**: Create derived features and risk indicators

**Features to Create**:
1. Temporal features (is_summer, season)
2. DO risk indicators (DO_critical, DO_stress)
3. Temperature features (temp_warm, temp_hot)
4. Combined stress indicators
5. Composite risk score (0-100)

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('../src')

# Import feature engineering functions
from feature_engineering import (
    create_temporal_features,
    create_do_risk_features,
    create_temperature_features,
    create_combined_stress_features,
    calculate_risk_score,
    create_site_aggregations
)

from data_processing import save_processed_data

print("✓ Libraries and modules imported successfully")

## 1. Load Cleaned Data

In [ ]:
# Load cleaned data
df = pd.read_csv('../data/processed/cleaned_water_quality.csv')

# Convert date columns
date_cols = [col for col in df.columns if 'date' in col.lower()]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print(f"Loaded {len(df):,} records")
print(f"Columns: {list(df.columns[:10])}...")

## 2. Identify Key Columns

Map standardized names to actual column names.

In [ ]:
# Find key columns
do_cols = [col for col in df.columns if 'do' in col.lower() or 'oxygen' in col.lower()]
temp_cols = [col for col in df.columns if 'temp' in col.lower()]
ph_cols = [col for col in df.columns if 'ph' in col.lower()]
flow_cols = [col for col in df.columns if 'flow' in col.lower()]
site_cols = [col for col in df.columns if 'site' in col.lower()]

print("Column mapping:")
print(f"  DO: {do_cols}")
print(f"  Temperature: {temp_cols}")
print(f"  pH: {ph_cols}")
print(f"  Flow: {flow_cols}")
print(f"  Site: {site_cols}")

# Set primary columns (adjust as needed)
do_col = do_cols[0] if do_cols else None
temp_col = temp_cols[0] if temp_cols else None
ph_col = ph_cols[0] if ph_cols else None
flow_col = flow_cols[0] if flow_cols else None
site_col = site_cols[0] if site_cols else None
date_col = date_cols[0] if date_cols else None

## 3. Create Temporal Features

Add season, summer flag, and other time-based features.

In [ ]:
# Create temporal features
if date_col:
    df = create_temporal_features(df, date_column=date_col)
    
    print("\nTemporal features created:")
    if 'season' in df.columns:
        print(df['season'].value_counts())
    if 'is_summer' in df.columns:
        print(f"\nSummer records: {df['is_summer'].sum():,} ({df['is_summer'].mean()*100:.1f}%)")
else:
    print("No date column found - skipping temporal features")

## 4. Create DO Risk Features

Binary and categorical indicators for dissolved oxygen stress.

In [ ]:
# Create DO risk features
if do_col:
    df = create_do_risk_features(df, do_column=do_col)
    
    print("\nDO Risk Features:")
    if 'DO_category' in df.columns:
        print(df['DO_category'].value_counts())
    if 'DO_critical' in df.columns:
        print(f"\nCritical DO events: {df['DO_critical'].sum():,}")
else:
    print("No DO column found - skipping DO features")

## 5. Create Temperature Features

Temperature-based risk categories.

In [ ]:
# Create temperature features
if temp_col:
    df = create_temperature_features(df, temp_column=temp_col)
    
    print("\nTemperature Features:")
    if 'temp_category' in df.columns:
        print(df['temp_category'].value_counts())
    if 'temp_warm' in df.columns:
        print(f"\nWarm events (>25°C): {df['temp_warm'].sum():,}")
    if 'temp_hot' in df.columns:
        print(f"Hot events (>28°C): {df['temp_hot'].sum():,}")
else:
    print("No temperature column found - skipping temperature features")

## 6. Create Combined Stress Features

Identify compound stress events (high temp + low DO).

In [ ]:
# Create combined stress features
if do_col and temp_col:
    df = create_combined_stress_features(df, 
                                         do_column=do_col,
                                         temp_column=temp_col,
                                         flow_column=flow_col)
    
    print("\nCombined Stress Events:")
    if 'stress_combo' in df.columns:
        stress_count = df['stress_combo'].sum()
        print(f"High temp + Low DO events: {stress_count:,} ({stress_count/len(df)*100:.2f}%)")
    if 'extreme_stress' in df.columns:
        extreme_count = df['extreme_stress'].sum()
        print(f"Extreme stress events: {extreme_count:,}")
else:
    print("Missing required columns for combined stress features")

## 7. Calculate Composite Risk Score

Create a 0-100 risk score combining multiple factors.

In [ ]:
# Calculate risk score
if do_col and temp_col:
    df = calculate_risk_score(df,
                              do_column=do_col,
                              temp_column=temp_col,
                              ph_column=ph_col,
                              flow_column=flow_col)
    
    print("\nRisk Score Statistics:")
    if 'risk_score' in df.columns:
        print(df['risk_score'].describe())
        print("\nRisk Category Distribution:")
        print(df['risk_category'].value_counts())
else:
    print("Missing required columns for risk score calculation")

## 8. Visualize Risk Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'risk_score' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram
    axes[0].hist(df['risk_score'].dropna(), bins=50, color='coral', edgecolor='black', alpha=0.7)
    axes[0].axvline(df['risk_score'].mean(), color='blue', linestyle='--', linewidth=2, 
                   label=f"Mean: {df['risk_score'].mean():.1f}")
    axes[0].axvline(df['risk_score'].median(), color='red', linestyle='--', linewidth=2,
                   label=f"Median: {df['risk_score'].median():.1f}")
    axes[0].set_xlabel('Risk Score', fontweight='bold')
    axes[0].set_ylabel('Frequency', fontweight='bold')
    axes[0].set_title('Risk Score Distribution', fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Category counts
    if 'risk_category' in df.columns:
        category_order = ['Low', 'Moderate', 'High', 'Very High', 'Extreme']
        category_counts = df['risk_category'].value_counts().reindex(category_order, fill_value=0)
        colors = ['green', 'yellow', 'orange', 'orangered', 'darkred']
        
        axes[1].bar(range(len(category_counts)), category_counts.values,
                   color=colors, edgecolor='black', alpha=0.7)
        axes[1].set_xticks(range(len(category_counts)))
        axes[1].set_xticklabels(category_order, rotation=45)
        axes[1].set_ylabel('Count', fontweight='bold')
        axes[1].set_title('Risk Category Distribution', fontweight='bold')
        axes[1].grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('../outputs/figures/06_risk_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()

## 9. Create Site-Level Aggregations

In [ ]:
# Create site aggregations
if site_col and do_col and temp_col:
    site_stats = create_site_aggregations(df,
                                          site_column=site_col,
                                          metrics=[do_col, temp_col])
    
    print("\nSite-level statistics:")
    print(site_stats.head(10))
    
    # Save site stats
    site_stats.to_csv('../data/processed/site_statistics.csv', index=False)
    print("\n✓ Site statistics saved to data/processed/site_statistics.csv")
else:
    print("Cannot create site aggregations - missing required columns")

## 10. Save Featured Dataset

In [ ]:
# Display all new features
new_features = [
    'season', 'is_summer', 'is_winter',
    'DO_critical', 'DO_stress', 'DO_optimal', 'DO_category',
    'temp_warm', 'temp_hot', 'temp_category',
    'stress_combo', 'flow_flag',
    'risk_score', 'risk_category'
]

available_features = [f for f in new_features if f in df.columns]

print("\nFeatures created:")
for feat in available_features:
    print(f"  ✓ {feat}")

print(f"\nTotal features: {len(available_features)}")
print(f"Total columns in dataset: {len(df.columns)}")

In [ ]:
# Save featured dataset
output_path = '../data/processed/water_quality_with_features.csv'
save_processed_data(df, output_path)

print("\n" + "="*60)
print("FEATURE ENGINEERING COMPLETE")
print("="*60)
print(f"Records: {len(df):,}")
print(f"Total features: {len(df.columns)}")
print(f"Output: {output_path}")
print("="*60)

## Next Steps

Proceed to **Notebook 04: Visualization** to:
- Create publication-quality plots
- Generate comprehensive dashboard
- Visualize risk patterns